In [ ]:
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, KFold
from catboost import CatBoostClassifier
%pip install kagglehub catboost xgboost tqdm -q


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

df_q3 = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df_q3.head()

In [ ]:
# Task 3: Write your code here:
df_q3.info()

In [ ]:
# Task 4: Write your code here:
df_q3.describe()

In [ ]:
# Task 1: Write your code here:

df_q3.isnull().sum()

num_cols = df_q3.select_dtypes(include=['number']).columns
cat_cols = df_q3.select_dtypes(include=['object']).columns

df_q3[num_cols] = df_q3[num_cols].fillna(df_q3[num_cols].median())

for col in cat_cols:
    df_q3[col] = df_q3[col].fillna(df_q3[col].mode()[0])


In [ ]:
# Task 2: Write your code here:
# Task 2: Check and remove duplicates
print("Duplicates:", df_q3.duplicated().sum())
df_q3 = df_q3.drop_duplicates()


In [ ]:
# Task 3: Write your code here:
# Task 3: Encode categorical variables (One Hot Encoding)
cat_cols = df_q3.select_dtypes(include=['object']).columns
df_q3 = pd.get_dummies(df_q3, columns=cat_cols, drop_first=True)


In [ ]:
# Task 4: Write your code here:
# Task 4: Apply feature scaling to numerical features (StandardScaler)
target_col = 'Target'  # change if your target column name is different

X = df_q3.drop(columns=[target_col])
y = df_q3[target_col]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 5: Write your code here:
# Task 5: Check target imbalance
y.value_counts(normalize=True)


In [ ]:
# Task 1: Write your code here:
# Task 1: Split the dataset into features (X) and target (y)
target_col = 'Target'  # change if your target column name is different

X = df_q3.drop(columns=[target_col])
y = df_q3[target_col]


In [ ]:
# Task 2,3,4,5: Write your code here:
# Task 2,3,4,5: StratifiedKFold + CatBoostClassifier + (Accuracy or F1) + average score

scores = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_seed=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    # Choose ONE metric:
    score = f1_score(y_val, y_pred)         # use this if imbalanced
    # score = accuracy_score(y_val, y_pred) # use this if balanced

    scores.append(score)

print("Scores per fold:", scores)
print("Average score:", sum(scores) / len(scores))


In [ ]:
# Task 1: Write your code here:


importances = model.get_feature_importance()
feature_names = X.columns

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'].head(20), importance_df['importance'].head(20))
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:


golden_feature = importance_df.iloc[0]['feature']
print("Golden Feature:", golden_feature)


In [ ]:
# Task Bonus: Write your code here:


X_golden = X[[golden_feature]]

full_scores = []
golden_scores = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Full model CV score
for train_idx, val_idx in cv.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    full_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_seed=42)
    full_model.fit(X_train, y_train)
    y_pred = full_model.predict(X_val)

    full_scores.append(f1_score(y_val, y_pred))

# Golden-feature-only model CV score
for train_idx, val_idx in cv.split(X_golden, y):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    golden_model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_seed=42)
    golden_model.fit(X_train, y_train)
    y_pred = golden_model.predict(X_val)

    golden_scores.append(f1_score(y_val, y_pred))

print("Full Model Avg F1:", sum(full_scores) / len(full_scores))
print("Golden Feature Only Avg F1:", sum(golden_scores) / len(golden_scores))
